In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import random
from features import limpiar_features_no_necesarias, transformar_features, transformar_target_a_estrellas, es_parte_de_saga, normalize_format

## 1. Comienzo por filtrar el dataset:

### 1.1 Primero saco las muestras de paises non-US:

In [ ]:
chunks = pd.read_json("goodreads_books.json", lines=True, chunksize=100_000)

us_list = []
for chunk in chunks:
    us_list.append(chunk[chunk["country_code"] == "US"])

df_us = pd.concat(us_list, ignore_index=True)
print(df_us.shape)


### 1.2 Ahora me tengo que quedar nomas con los libros que son en ingles:

Consideramos los libros en ingles los que tienen "languade_code" = ["en", "eng", "en-US", "en-GB", "en-CA", "enm", "ang", "sco", "gem"]

Las ultimas 4 opciones son ingles, pero antiguos (afectara en el rating del libro?... Acaso como a las personas de hoy en dia les cosatara mas atenderlo y por eso le daran bajo rating?...)

Tambien me quedo con las muestras con ese valor NaN, porque aun no podemos decidir si descartarlos o no.

In [ ]:
valid_langs = {"en", "eng", "en-us", "en-gb", "en-ca", "enm", "ang", "sco", "gem"}
lang = df_us["language_code"].fillna("").str.strip().str.lower()
df_us_eng = df_us[lang.isin(valid_langs) | (lang == "")]
print(df_us_eng.shape)

Lo guardo en un json:

In [ ]:
import json
output_path = "goodreads_us_english.json"
chunk_size = 10000
with open(output_path, "w", encoding="utf-8") as f:
    for start in range(0, len(df_us_eng), chunk_size):
        chunk = df_us_eng.iloc[start:start + chunk_size]
        records = chunk.where(pd.notna(chunk), None).to_dict(orient="records")
        for record in records:
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
print("Guardado en", output_path)

Ahora abro el nuevo archivo que acabo de crear, asi trabajo sobre ese:

In [2]:
chunks2 = pd.read_json("goodreads_us_english.json", lines=True, chunksize=100_000)

chunk_list = []
for chunk in chunks2:
    chunk_list.append(chunk)

df_us_en = pd.concat(chunk_list, ignore_index=True)
print(df_us_en.shape)

(1925872, 29)


## 2. Reorganizo valores de features (breve inicio del preprocesamiento):

### 2.1 Comienzo por reagrupar los valores de "format"

In [3]:
# format_uniques = df_us_en["format"].unique()
# for v in format_uniques:
#     print(v)

In [4]:
# df_us_en["format"].value_counts().head(50)

In [5]:
mixed = df_us_en["format"].str.contains(
    "paperback.*ebook|ebook.*paperback|kindle.*paperback|paperback.*kindle",
    case=False,
    na=False
)

print(mixed.sum())

224


In [7]:
#Le aplico la funcion para reagrupar:
if "format_group" not in df_us_en.columns:
    df_us_en["format_group"] = df_us_en["format"].apply(normalize_format)

print(df_us_en["format_group"].value_counts())

format_group
PAPERBACK    683447
UNKNOWN      591456
EBOOK        293266
HARDCOVER    281921
AUDIO         67250
OTHER          7742
COMIC           790
Name: count, dtype: int64


In [8]:
df_us_en["format_group"].value_counts(normalize=True) * 100

format_group
PAPERBACK    35.487665
UNKNOWN      30.711075
EBOOK        15.227699
HARDCOVER    14.638616
AUDIO         3.491925
OTHER         0.402000
COMIC         0.041020
Name: proportion, dtype: float64

Mas o menos el 99.59% de las muestars quedaron clasificadas en categorias concretas --> esto esta muy bueno!
Y podemos ver que hay muhcos UNKOWNS, lo que signfica que el usuario no compleot la info

### 2.2 Convertimos "series" en una feature binaria

In [ ]:
#is_series: feature binaria, true si es parte de una saga
df_us_en["is_series"] = df_us_en["series"].apply(es_parte_de_saga)
df_us_en = df_us_en.drop(columns = ["series"])

## 3. Analizo features (EDA):

### 3.1 Analisando el target

In [ ]:
df_us_en["average_rating"] = pd.to_numeric(df["average_rating"], errors='coerce')
print(df_us_en["average_rating"].isna().sum())
print(df_us_en["average_rating"].max())
print(df_us_en["average_rating"].min())

En este sample, hay XX libros sin rating. Corresponde al XXX%. Podemos eliminar esos libros. Notamos aparte que no hay outliers. --> SIGUE EN PIE ESTO CON TODO EL DATASET???

In [ ]:
df_us_en = df_us_en.dropna(subset=["average_rating"])
target_regresion = df_us_en["average_rating"]
target_clasificacion = transformar_target_a_estrellas(target_regresion)

In [ ]:
df_us_en["average_rating"].describe()

Como lo van a confirmar los graficos mas adelante, la mayoria de ratings se encuentran entre 3.5 y 4.5. 

In [ ]:
df_us_en["stars"].value_counts(normalize=True).sort_index()

El desbalanceo de clases se hace aun mas presente en el target para clasificacion.

In [ ]:
target_clasificacion_balanceado = pd.qcut(df_us_en["average_rating"], q=5)
print(target_clasificacion_balanceado.value_counts().sort_index())

Esta estrategia permitiria mantener 5 clases balanceadas, pero perdemos la interpretabilidad de las 5 estrellas. 

In [ ]:
plt.hist(df_us_en["average_rating"], bins=50)
plt.xlabel("Average Rating")
plt.ylabel("Cantidad de libros")
plt.title("Distribución de ratings")
plt.show()

plt.boxplot(df_us_en["average_rating"])
plt.ylabel("Average Rating")
plt.show()

In [ ]:
df_us_en["stars"] = df_us_en["average_rating"].round().astype(int)
df_us_en["stars"].value_counts().sort_index().plot(kind="bar")

plt.title("Distribución de estrellas")
plt.show()

### 3.2 Veo cuantas muestras tienen el laguage_code incompleto:

In [9]:
missing = df_us_en["language_code"].isna().sum() #Aplica para NaN y None
empty_str = (df_us_en["language_code"] == "").sum()
print("NaN:", missing)
print("Cadena vacía:", empty_str)
print("Total vacío:", missing + empty_str)

NaN: 0
Cadena vacía: 1059663
Total vacío: 1059663


### 3.3 ¿El formato del libro tiene alguna relacion con el rating?

In [12]:
#df_us_en.groupby("format_group").apply(lambda x: pd.to_numeric(x["average_rating"], errors="coerce").mean())

In [ ]:
df_us_en.groupby("format_group")["average_rating"].agg(["count","mean","std"])

### 3.4 La variable is_ebook, es util teniendo en cuenta que vamos a hacer un one hot encoding a la nueva feature de format_group?

### 3.5 "series"

In [ ]:
df_us_en["series"].str.len().value_counts().sort_index()

La feature "series" contiene una lista. Si la lista esta vacia el libro no es parte de una saga. No tiene valores Nan o que no sean una lista. --> esto ultimo se mantiene con este nuevo dataset???

### 3.6 "publication_year" y "num_pages"

In [ ]:
df_us_en[["publication_year","num_pages"]].describe()
df_us_en["publication_year"] = pd.to_numeric(df_us_en["publication_year"], errors='coerce')
df_us_en["num_pages"] = pd.to_numeric(df_us_en["num_pages"],errors='coerce')
df_us_en[["publication_year","num_pages"]].describe()

print(df_us_en["num_pages"].isna().sum())
print(df_us_en["publication_year"].isna().sum())

Noto que hay que tratar con varios nan en publication year y num pages. Hay que decidir como tratar eso.

### 3.7 Buscando la feature "genero"

In [ ]:
for i in range(5):
    print(f"Libro {i}")
    print(df_us_en["popular_shelves"].iloc[i])
    print()

from collections import Counter

counter = Counter()

for shelves in df_us_en["popular_shelves"]:
    for shelf in shelves:
        counter[shelf["name"]] += 1

In [ ]:
counter.most_common(200)

In [ ]:
genres = ["fiction", "romance", "contemporary", "adult", "non-fiction", "fantasy", "mystery", "novels", "history", "nonfiction", "historical", "young-adul", "novel", "historical-fiction", "adult-fiction", "adventure", "literature", "classics", "thriller", "ya", "suspense", "paranormal", "science-fiction", "crime", "drama", "sci-fi", "contemporary-romance", "general-fiction", "contemporary-fiction", "childrens", "children", "mystery-thriller", "classic", "biography", "sci-fi-fantasy", "horror", "mysteries", "magic", "children-s", "mystery-suspense", "philosophy", "science", "erotica", "literary", "fantasy-sci-fi", "religion", "scifi", "politics", "memoir", "paranormal-romance", "thrillers", "psychology", "ya-fiction", "crime-fiction", "juvenile"]

## 4. Continuamos el preprocesamiento (se empezo un poco antes del EDA):

In [ ]:
df_procesado = transformar_features(df_us_en)
df_entrenamiento = limpiar_features_no_necesarias(df_procesado)

# TODO

saga func clari , incluye rautng graf
autores agrupar para luego inflkuye rating graf 